In [ ]:
import gc
import tqdm
import pandas as pd
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from default_risk.scripts.feature_cleaner import clean_noise_from_feature_importance
from default_risk.scripts.feature_cleaner import creating_criteria


In [ ]:


df = pd.read_csv(cfg.ARTIFACTS_DIR / "feature-auc-impact.csv")


In [ ]:
inhert_features = df[
    (df['auc_impact'] == 0) & (df['std_impact'] == 0)
]['feature_dropped']

# Imprimimos los nombres
print(inhert_features.tolist())

['flag_own_car', 'amt_goods_price_is_missing', 'have_sentinel_value_days_employed', 'own_car_age_is_missing', 'flag_emp_phone', 'flag_cont_mobile', 'ext_source_1_is_missing', 'ext_source_2_is_missing', 'ext_source_3_is_missing', 'info_of_social_circule_is_missing', 'flag_document_9', 'flag_document_13', 'flag_document_16', 'client_without_querys', 'organization_type_Advertising', 'organization_type_Electricity', 'organization_type_Emergency', 'organization_type_Housing', 'organization_type_Industry: type 1', 'organization_type_Industry: type 7', 'organization_type_Other industry', 'organization_type_Other trade', 'organization_type_Postal', 'organization_type_Security', 'organization_type_Services', 'organization_type_Trade: type 2', 'organization_type_Trade: type 6', 'organization_type_Transport: type 2', 'organization_type_XNA', 'bureau_credit_currency_loan_1']


In [20]:
pd.options.display.float_format = '{:.6f}'.format
pd.set_option('display.max_rows', None)
df_ordenado = df.sort_values(by='auc_impact', ascending=True)
print(df_ordenado[df['std_impact'] > 0])

                                  feature_dropped  auc_impact  std_impact
19                              days_registration   -0.002085    0.000686
48                                 housetype_mode   -0.001556    0.000293
7                                     amt_annuity   -0.001081    0.000547
18                                  days_employed   -0.000976    0.000877
145                                ext_source_std   -0.000948    0.000041
0                              name_contract_type   -0.000942    0.000569
49                             wallsmaterial_mode   -0.000923    0.000596
27                                occupation_type   -0.000847    0.000157
78                                      kui_ratio   -0.000833    0.000072
57                                flag_document_3   -0.000793    0.000067
120                  organization_type_Restaurant   -0.000749    0.000015
76                                 ratio_debt_age   -0.000710    0.000457
79                              ratio_

/var/folders/q_/4qptmrj56q78qnnfbxh4fmdm0000gn/T/ipykernel_74462/2070260977.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  print(df_ordenado[df['std_impact'] > 0])


2026-06-26 10:46:03,271 - INFO     - Executing shutdown due to inactivity...
2026-06-26 10:46:03,660 - INFO     - Executing shutdown...
2026-06-26 10:46:03,665 - INFO     - Not running with the Werkzeug Server, exiting by searching gc for BaseWSGIServer
